# Lab · RAG retrieval, made visible (free Kaggle T4)

**Layer 3 · Data.** The heart of RAG is **retrieval**: the model can only answer
from the chunks you load into the prompt. This lab makes that visible on a **real
public Kubernetes Q&A dataset**, and walks the three retrieval knobs:

1. **See which chunks get loaded** for a question
2. **top-k** — how many chunks to load
3. **chunk size** — how big each piece is
4. **reranker** — re-sort the candidates so the best one lands first

…then we feed the retrieved chunks to **Qwen2.5-3B** for a grounded, cited answer.

**Settings:** Accelerator = **GPU T4 x2** (not P100), Internet = **On**, then
**Run All**. Most cells are CPU-only and instant; only the final answer uses the GPU.

*Data: [`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs) — real, clean k8s Q&A.*

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')                 # silence library UserWarnings / deprecations
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'     # quiet model-load reports + generation-flag notes
os.environ['HF_HUB_VERBOSITY'] = 'error'           # quiet HF Hub 'unauthenticated requests' notice
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
for _n in ('huggingface_hub', 'sentence_transformers', 'transformers', 'datasets'):
    logging.getLogger(_n).setLevel(logging.ERROR)

!pip install -q faiss-cpu sentence-transformers datasets 2>/dev/null

import faiss, textwrap
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

## 1 · A real public dataset (Phase 1 · Sources)

We pull a public Kubernetes Q&A set straight from the Hugging Face Hub — ~500 real
question/answer pairs, each tagged with a `topic` and `difficulty`. We'll index the
**answers** as our knowledge chunks.

In [ ]:
from datasets import load_dataset
ds = load_dataset('ItshMoh/kubernetes_qa_pairs', split='train')
df = ds.to_pandas()
print('rows:', len(df), '| columns:', list(df.columns))
print('topics:', df['topic'].nunique())
df[['question', 'answer', 'topic']].head(3)

## 2 · Embed + index (Phases 3–4)

Every answer becomes a **chunk** → a vector via **`bge-small`** (tiny model, run on
**CPU** — instant here). We store the vectors in a flat **FAISS** index (exact
cosine search; you'd only need HNSW past ~100k vectors). The query later uses the
**same** embedder — the #1 RAG rule.

In [ ]:
EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder = SentenceTransformer(EMB_ID, device='cpu')

chunks = [{'id': f'qa-{i}', 'text': r['answer'], 'question': r['question'], 'topic': r['topic']}
          for i, r in enumerate(ds) if (r['answer'] or '').strip()]
vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=128).astype('float32')
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

def retrieve(query, k=3):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]

print('indexed', index.ntotal, 'answer-chunks')

## 3 · See which chunks get loaded

This is the whole game: for a question, retrieval returns the nearest chunks, and
**those — and only those — go into the prompt.** Let's print them, with their
similarity score and topic, so you can *see* what the model will be allowed to read.

In [ ]:
def show_hits(query, k=3):
    print(f'QUERY: {query}\n')
    print(f'--- top-{k} chunks loaded into the prompt ---')
    for rank, h in enumerate(retrieve(query, k=k), 1):
        print(f"#{rank}  score={h['score']:.3f}  topic={h['topic']}")
        print('    Q: ' + h['question'])
        print('    A: ' + textwrap.fill(h['text'], 84).replace('\n', '\n       '))

show_hits('What does the kube-scheduler consider when selecting a node for a Pod?', k=3)

## 4 · top-k — how many chunks to load

**top-k** is the dial for how much context you pull per question.

- too small (k=1): if the best chunk is slightly off, the model has nothing to lean on
- too big (k=10): the real answer gets buried in noise, and you pay for the extra tokens

Watch how the loaded set grows (and drifts off-topic) as k increases.

In [ ]:
q = 'What does the kube-scheduler consider when selecting a node for a Pod?'
for k in (1, 3, 6):
    hits = retrieve(q, k=k)
    print(f'k={k}: ' + ' | '.join(f"{h['topic']}({h['score']:.2f})" for h in hits))

## 5 · Chunk size — how big each piece is

We indexed each answer whole. But long documents must be **split into chunks**
first. Chunk size is a trade-off:

- **small** chunks → pinpoint retrieval, but a fact split across two chunks can be lost
- **large** chunks → full context, but the matching signal gets diluted

We take one topic's text, split it small vs large, and retrieve the same query
against each — see how the *granularity* of what loads changes.

In [ ]:
topic = df['topic'].value_counts().index[0]
long_text = ' '.join(df[df['topic'] == topic]['answer'].tolist())
print(f'topic "{topic}": {len(long_text.split())} words of source text\n')

def word_chunks(text, size, overlap):
    w = text.split(); step = size - overlap
    return [' '.join(w[i:i+size]) for i in range(0, len(w), step) if w[i:i+size]]

def mini_index(text, size, overlap):
    cs = word_chunks(text, size, overlap)
    v = embedder.encode(cs, normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    ix = faiss.IndexFlatIP(v.shape[1]); ix.add(v)
    return cs, ix

def top_chunk(cs, ix, query):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    _, ii = ix.search(qv, 1); return cs[int(ii[0][0])]

small_cs, small_ix = mini_index(long_text, size=20, overlap=4)
big_cs,   big_ix   = mini_index(long_text, size=80, overlap=16)
print(f'same text -> {len(small_cs)} small chunks  vs  {len(big_cs)} big chunks\n')

query = f'Tell me about {topic.lower()} in Kubernetes'
print('SMALL chunk retrieved (precise, may lack context):')
print('  ' + textwrap.fill(top_chunk(small_cs, small_ix, query), 86).replace('\n', '\n  '))
print('\nBIG chunk retrieved (more context, less precise):')
print('  ' + textwrap.fill(top_chunk(big_cs, big_ix, query), 86).replace('\n', '\n  '))

## 6 · Reranker — put the best chunk first

Vector search is fast but blunt: it compares pre-computed vectors. A **cross-encoder
reranker** reads the query and each chunk *together* and re-scores them — much more
accurate. The pattern: **over-fetch** a generous top-k cheaply, then rerank and keep
the best few. Watch a chunk jump to #1.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

# Vector search alone grabs a loosely-related "images" chunk; the cross-encoder reads
# query+chunk together and promotes the real garbage-collection answer to the top.
q = 'How does Kubernetes clean up unused container images?'
cands = retrieve(q, k=10)                      # over-fetch cheaply
print('BEFORE rerank (vector order), top 3:')
for h in cands[:3]:
    print(f"  {h['score']:.3f}  {h['topic']}: {h['text'][:70]}...")

scores = reranker.predict([[q, h['text']] for h in cands])
for h, s in zip(cands, scores):
    h['rr'] = float(s)
reranked = sorted(cands, key=lambda h: h['rr'], reverse=True)
print('\nAFTER rerank (cross-encoder), top 3:')
for h in reranked[:3]:
    print(f"  rr={h['rr']:.2f}  {h['topic']}: {h['text'][:70]}...")

## 7 · Feed the chunks to the model → grounded, cited answer

Finally, take the reranked top chunks, put them in the prompt as CONTEXT, and let
**Qwen2.5-3B** answer *only* from them, with a citation. (This cell loads the LLM —
~6 GB on the T4.) We also show the **refusal** when the context can't answer.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto').eval()
print('LLM on', model.device)

SYS = ('You are a Kubernetes assistant. Use ONLY the CONTEXT to answer, and cite the chunk ids in [brackets]. '
       "Do not use any outside knowledge. If the answer is not in the CONTEXT, reply exactly: I don't know.")

def answer(query, k=10, keep=3):
    cands = retrieve(query, k=k)
    scores = reranker.predict([[query, h['text']] for h in cands])
    for h, s in zip(cands, scores): h['rr'] = float(s)
    hits = sorted(cands, key=lambda h: h['rr'], reverse=True)[:keep]
    ctx = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    text = tok.apply_chat_template(
        [{'role': 'system', 'content': SYS},
         {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {query}'}],
        tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip(), hits

ans, hits = answer('What is Role-Based Access Control (RBAC) in Kubernetes?')
print('loaded chunks:', [h['id'] for h in hits])
print('\nANSWER:\n', textwrap.fill(ans, 90))

In [ ]:
# refusal: nothing in a k8s corpus answers this
ans, _ = answer('What is the capital of France?')
print(textwrap.fill(ans, 90))

## Takeaways

- **Retrieval is the heart of RAG** — the model can only answer from the chunks you
  load. Print them (cell 3) and you can debug RAG; most failures are *retrieval*
  failures, not model failures.
- **top-k** = how many chunks to load. Small starves the model; big adds noise and
  cost. Start at 3–8.
- **chunk size** = how big each piece is. Small = precise but fragmented; large =
  full context but diluted.
- **reranker** = a cross-encoder that re-reads query+chunk together and puts the
  best one first. Over-fetch cheap, rerank, keep the top few — the biggest quality
  lever after chunking.
- Same embedder for docs and queries; flat FAISS is exact at this scale.

Note: on a *public* dataset the model often already knows the topic, so the value
here is **citations + retrieval quality**, not blocking hallucination — for the
private-data hallucination contrast, see the runbook version in git history.